<a href="https://colab.research.google.com/github/oluigifreire/EVChallenge2026-1CCA-Equipe7-Prompt-and-Artificial-Intelligence-Sprint1/blob/main/Chatbot_ChargeGrid_IA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
"""ChargeGrid Intelligence — v1
Dashboard gerado por IA + Chatbot Gerencial
GoodWe & FIAP
"""

!pip install openai plotly -q

import openai
import json
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from openai import OpenAI
from google.colab import userdata
from IPython.display import display, Markdown

client = OpenAI(api_key=userdata.get('OPENAI_API_KEY'))

# ════════════════════════════════════════════════════════════════════
# 1. SYSTEM PROMPT — injete aqui os dados do dia manualmente
#    Troque apenas os valores abaixo quando os dados mudarem
# ════════════════════════════════════════════════════════════════════

SYSTEM_PROMPT = """
# IDENTIDADE

Você é o assistente gerencial do ChargeGrid Intelligence, uma plataforma
desenvolvida pela GoodWe para gestão inteligente de eletropostos comerciais.

Seu papel é ser o primeiro contato do gestor do estabelecimento com os dados
de operação dos carregadores. Você entrega um briefing matinal completo,
identifica anomalias, e propõe ações concretas baseadas nos dados do dia
anterior.

Tom: objetivo, técnico mas acessível, direto. Você fala como um analista
sênior de operações — sem enrolação, sem respostas genéricas. Cada
recomendação sua tem uma justificativa baseada nos dados.

Você NÃO executa ações nos carregadores diretamente. Você recomenda.
A execução automática é responsabilidade do agente de controle em tempo
real (módulo futuro do ChargeGrid). Deixe isso claro quando relevante.

Formate sempre suas respostas em Markdown: use títulos (##), listas (-),
negrito (**) e tabelas quando relevante.

---

# CONTEXTO DO SISTEMA — DADOS DO DIA ANTERIOR

## Configuração do site
- Estabelecimento: Shopping (estacionamento coberto)
- Total de carregadores: 10x GoodWe HCA G2 11kW
- Capacidade máxima teórica: 110 kW
- Limite de demanda contratada: 80 kW
- Ultrapassar esse limite gera multa da concessionária e risco de queda de disjuntor

## Resumo do dia
- Total de sessões realizadas: 47
- Energia total entregue: 342 kWh
- Faturamento estimado: R$ 1.128,00
- Pico de demanda registrado: 78,4 kW (às 14h00)
- Percentual do limite atingido no pico: 98%
- Janela crítica: 14h00 às 15h30

## Consumo por hora (demanda média em kW)
08h: 11.0
09h: 18.0
10h: 25.0
11h: 33.0
12h: 41.0
13h: 52.0
14h: 78.4
15h: 71.0
16h: 65.0
17h: 58.0
18h: 49.0
19h: 38.0
20h: 27.0
21h: 19.0
22h: 8.0

## Status individual dos carregadores no momento do pico (14h30)
EV-01: 11.0 kW | 6 sessões | 38 kWh | Status: Pico crítico
EV-02: 11.0 kW | 5 sessões | 34 kWh | Status: Pico crítico
EV-03:  9.2 kW | 5 sessões | 31 kWh | Status: Atenção
EV-04: 11.0 kW | 6 sessões | 37 kWh | Status: Pico crítico
EV-05:  8.1 kW | 4 sessões | 28 kWh | Status: Atenção
EV-06: 11.0 kW | 5 sessões | 35 kWh | Status: Pico crítico
EV-07:  7.1 kW | 4 sessões | 26 kWh | Status: Normal
EV-08:  0.0 kW | 3 sessões | 22 kWh | Status: Ocioso
EV-08 ficou ocioso das 13h00 às 16h30 (3h30 sem sessão ativa durante janela de pico)
EV-09:  6.0 kW | 4 sessões | 24 kWh | Status: Normal
EV-10:  4.0 kW | 5 sessões | 27 kWh | Status: Atenção

---

# REGRAS DE NEGÓCIO

## Alertas de demanda
- Acima de 70 kW (87.5% do limite): alerta amarelo — atenção elevada
- Acima de 76 kW (95% do limite): alerta vermelho — risco crítico
- Acima de 80 kW: limite ultrapassado — registrar como incidente

## Throttling (recomendação)
Quando a demanda ultrapassar 76 kW:
- Identificar carregadores em carga máxima (11 kW)
- Reduzir cada um para 8 kW, mantendo total abaixo de 72 kW
- Prioridade: carregadores com menor tempo de sessão ativo

## Eficiência operacional
- Carregador ocioso por mais de 2h durante horário de pico = ineficiência a reportar
- Taxa de ocupação ideal no horário comercial (10h-20h): acima de 60%

---

# RESTRIÇÕES

- Não sugira ações que interrompam sessões ativas abruptamente sem aviso.
- Não afirme que o sistema executou throttling automaticamente.
- Não use linguagem alarmista desnecessária.
- Ao calcular faturamento perdido, foque nos horários de baixa demanda (08h-11h),
  não na subutilização durante o pico.
"""

# ════════════════════════════════════════════════════════════════════
# 2. EXTRAÇÃO DE DADOS VIA IA — o modelo lê o system prompt e
#    devolve JSON estruturado para o plotly gerar os gráficos
# ════════════════════════════════════════════════════════════════════

PROMPT_EXTRACAO = """
Leia o contexto do sistema abaixo e extraia os dados em JSON puro.
Não inclua explicações, markdown ou texto fora do JSON.
Retorne EXATAMENTE neste formato:

{
  "resumo": {
    "sessoes": <int>,
    "kwh_total": <float>,
    "faturamento": <float>,
    "pico_kw": <float>,
    "pico_hora": "<string>",
    "pico_percentual": <float>,
    "limite_kw": <int>
  },
  "consumo_hora": [
    {"hora": "<string>", "kw": <float>},
    ...
  ],
  "carregadores": [
    {"id": "<string>", "kw_pico": <float>, "sessoes": <int>, "kwh": <float>, "status": "<string>"},
    ...
  ]
}

Contexto:
""" + SYSTEM_PROMPT

def extrair_dados_json():
    resposta = client.chat.completions.create(
        model='gpt-4o-mini',
        messages=[{"role": "user", "content": PROMPT_EXTRACAO}],
        temperature=0
    )
    texto = resposta.choices[0].message.content.strip()
    # Remove blocos de código markdown se o modelo incluir
    texto = texto.replace('```json', '').replace('```', '').strip()
    return json.loads(texto)

# ════════════════════════════════════════════════════════════════════
# 3. GERAÇÃO DO DASHBOARD COM PLOTLY
# ════════════════════════════════════════════════════════════════════

CORES_STATUS = {
    'Pico crítico': '#E24B4A',
    'Atenção':      '#EF9F27',
    'Normal':       '#639922',
    'Ocioso':       '#B4B2A9'
}

def gerar_dashboard(dados):
    resumo      = dados['resumo']
    consumo     = dados['consumo_hora']
    carregadores = dados['carregadores']

    horas   = [c['hora'] for c in consumo]
    demanda = [c['kw'] for c in consumo]
    limite  = [resumo['limite_kw']] * len(horas)
    alerta  = [70] * len(horas)

    ids      = [c['id'] for c in carregadores]
    kw_pico  = [c['kw_pico'] for c in carregadores]
    sessoes  = [c['sessoes'] for c in carregadores]
    kwh      = [c['kwh'] for c in carregadores]
    status   = [c['status'] for c in carregadores]
    cores    = [CORES_STATUS.get(s, '#888780') for s in status]

    fig = make_subplots(
        rows=2, cols=2,
        subplot_titles=(
            'Demanda ao longo do dia (kW)',
            'Potência por carregador no pico (kW)',
            'kWh entregues por carregador',
            'Sessões realizadas por carregador'
        ),
        vertical_spacing=0.18,
        horizontal_spacing=0.10
    )

    # Gráfico 1 — Linha de demanda
    fig.add_trace(go.Scatter(
        x=horas, y=demanda,
        mode='lines+markers',
        name='Demanda (kW)',
        line=dict(color='#D85A30', width=2.5),
        marker=dict(
            size=[14 if d >= 76 else 7 for d in demanda],
            color=['#A32D2D' if d >= 76 else '#D85A30' for d in demanda],
            symbol=['diamond' if d >= 76 else 'circle' for d in demanda]
        ),
        fill='tozeroy',
        fillcolor='rgba(216,90,48,0.08)',
        hovertemplate='%{x}: %{y} kW<extra></extra>'
    ), row=1, col=1)

    fig.add_trace(go.Scatter(
        x=horas, y=limite, mode='lines',
        name=f'Limite ({resumo["limite_kw"]} kW)',
        line=dict(color='#E24B4A', width=1.5, dash='dash'),
        hoverinfo='skip'
    ), row=1, col=1)

    fig.add_trace(go.Scatter(
        x=horas, y=alerta, mode='lines',
        name='Alerta (70 kW)',
        line=dict(color='#EF9F27', width=1, dash='dot'),
        hoverinfo='skip'
    ), row=1, col=1)

    fig.add_annotation(
        x=resumo['pico_hora'], y=resumo['pico_kw'],
        text=f"⚠ Pico {resumo['pico_kw']} kW ({resumo['pico_percentual']}%)",
        showarrow=True, arrowhead=2,
        arrowcolor='#A32D2D', ax=45, ay=-35,
        font=dict(color='#A32D2D', size=11),
        row=1, col=1
    )

    # Gráfico 2 — Barras horizontais por carregador
    fig.add_trace(go.Bar(
        x=kw_pico, y=ids,
        orientation='h',
        name='kW no pico',
        marker_color=cores,
        text=[f'{v} kW' for v in kw_pico],
        textposition='outside',
        hovertemplate='%{y}: %{x} kW<extra></extra>',
        showlegend=False
    ), row=1, col=2)

    fig.add_vline(
        x=11, line_dash='dash',
        line_color='#B4B2A9', line_width=1,
        annotation_text='Máx 11kW',
        annotation_font_size=10,
        row=1, col=2
    )

    # Gráfico 3 — kWh por carregador
    fig.add_trace(go.Bar(
        x=ids, y=kwh,
        name='kWh entregues',
        marker_color=cores,
        text=[f'{v}' for v in kwh],
        textposition='outside',
        hovertemplate='%{x}: %{y} kWh<extra></extra>',
        showlegend=False
    ), row=2, col=1)

    # Gráfico 4 — Sessões por carregador
    fig.add_trace(go.Bar(
        x=ids, y=sessoes,
        name='Sessões',
        marker_color=cores,
        text=[f'{v}' for v in sessoes],
        textposition='outside',
        hovertemplate='%{x}: %{y} sessões<extra></extra>',
        showlegend=False
    ), row=2, col=2)

    fig.update_layout(
        title=dict(
            text='ChargeGrid Intelligence — Relatório do Dia Anterior',
            font=dict(size=16), x=0.5
        ),
        height=650,
        paper_bgcolor='white',
        plot_bgcolor='white',
        font=dict(family='Arial', size=12, color='#444441'),
        legend=dict(
            orientation='h', yanchor='bottom',
            y=1.02, xanchor='right', x=1,
            font=dict(size=11)
        ),
        margin=dict(t=80, b=40, l=40, r=40)
    )

    fig.update_xaxes(showgrid=False)
    fig.update_yaxes(gridcolor='rgba(136,135,128,0.15)', gridwidth=0.5)
    fig.update_yaxes(range=[0, 100], row=1, col=1)

    # Tabela de resumo
    display(Markdown(f"""
---
## ⚡ ChargeGrid Intelligence — Dashboard Gerencial
**GoodWe | Relatório do dia anterior**

| Métrica | Valor |
|---|---|
| Total de sessões | {resumo['sessoes']} |
| Energia entregue | {resumo['kwh_total']} kWh |
| Faturamento estimado | R$ {resumo['faturamento']:,.2f} |
| Pico de demanda | **{resumo['pico_kw']} kW às {resumo['pico_hora']} ({resumo['pico_percentual']}% do limite)** |
| Limite contratado | {resumo['limite_kw']} kW |

> ⚠ **Alerta:** Demanda atingiu {resumo['pico_percentual']}% do limite contratado.
> Risco de multa por ultrapassagem e queda de disjuntor.
---
"""))

    fig.show()

# ════════════════════════════════════════════════════════════════════
# 4. CHATBOT — briefing automático + loop de conversa
# ════════════════════════════════════════════════════════════════════

def chamar_modelo(mensagens):
    resposta = client.chat.completions.create(
        model='gpt-4o-mini',
        messages=mensagens
    )
    return resposta.choices[0].message.content

def exibir(texto):
    display(Markdown(texto))

# ── EXECUÇÃO PRINCIPAL ───────────────────────────────────────────────────────

print("Extraindo dados do sistema...")
dados = extrair_dados_json()

print("Gerando dashboard...\n")
gerar_dashboard(dados)

msgs = [{"role": "system", "content": SYSTEM_PROMPT}]

exibir("\n---\n*Gerando briefing do dia anterior...*\n")
msgs.append({"role": "user", "content": "BRIEFING_MATINAL"})
briefing = chamar_modelo(msgs)
msgs.append({"role": "assistant", "content": briefing})
exibir(briefing)

exibir("\n---\n*Digite sua pergunta abaixo. Para encerrar, digite* `sair`.\n")

while True:
    pergunta = input("Você: ").strip()

    if not pergunta:
        continue

    if pergunta.lower() in ["sair"]:
        exibir("**ChargeGrid Intelligence a disposição. Até amanhã chefe!**")
        break

    msgs.append({"role": "user", "content": pergunta})
    resposta = chamar_modelo(msgs)
    msgs.append({"role": "assistant", "content": resposta})
    exibir(f"\n**ChargeGrid:**\n\n{resposta}\n")

PERGUNTAS:

*   O que exatamente causou o pico de ontem às 14h e qual foi o risco real para o shopping?

*   Se o mesmo padrão se repetir hoje às 14h, o que o ChargeGrid recomenda fazer e em quais carregadores?

*   Algum carregador teve desempenho abaixo do esperado ontem?

*   Nos horários de baixa demanda entre 08h e 11h, qual foi a capacidade desperdiçada e quanto isso representa em faturamento potencial?

*   Com base nos dados de ontem, que ação preventiva devo tomar antes das 13h hoje para evitar o pico crítico?